# Advanced AI - Assignment 6: Random Forests

Work through the sections below in order. The implementation lives in `advanced_ai/forest/random_forest.py`, reusing the decision tree from Lab 5, and uses NumPy only for the core algorithm.

## Core Equations And Rules

Use these definitions consistently in your implementation.

- Bootstrap sample $(X^{(b)}, y^{(b)})$: $n$ draws uniformly with replacement from $(X, y)$.
- OOB probability:

$$
P\bigl((x_i,y_i)\notin (X^{(b)},y^{(b)})\bigr)=\Bigl(1-\frac{1}{n}\Bigr)^{\!n}\approx\frac{1}{e}\approx 0.368.
$$

- Bagging prediction (majority vote):

$$
\hat{y}(x) = \arg\max_{c\in\{1,\dots,C\}}\, \sum_{b=1}^{B}\mathbf{1}\{\hat{y}^{(b)}(x)=c\}.
$$

- Feature subset size (classification): $m = \lfloor\sqrt{d}\rfloor$.
- Tree impurity and split: same as in Lab 5 (Gini impurity $I_G$, impurity decrease $\Delta I$, threshold search).
- Feature importance for feature $j$:

$$
I_j = \frac{1}{B}\sum_{b=1}^{B}\; \sum_{\substack{m\in T_b\\ \text{feature }m=j}} \frac{n_m}{n}\,\Delta I.
$$

## Step 1: Data files and preparation

Prepare the instructor-provided data bundles before running the notebook:

```bash
bash scripts/prepare_all_data.sh
bash scripts/check_prepare_all_data.sh
```

Expected prepared files:
- `datasets/toy_forest.npz`
- `datasets/iris_forest.npz`
- `datasets/forest_study.npz`

The scripts use the conda environment `ai_courses`.

In [ ]:
from advanced_ai.config import *
from advanced_ai.task_utils import require_file
from advanced_ai.data_utils import load_toy_bundle, load_iris_bundle, load_forest_study_bundle
from advanced_ai.trees.decision_tree import DecisionTreeClassifier
from advanced_ai.forest.random_forest import RandomForestClassifier

print('Imports ready.')

## Roadmap

Work in this order:

1. Prepare and verify the datasets.
2. Complete the hand calculations for bagging votes and OOB indices.
3. Implement the random forest core in `advanced_ai/forest/random_forest.py`.
4. Run the self-checks.
5. Run the toy, Iris, and OOB/importance task runners.

Do not skip the self-checks before running the experiments.

## Step 2: Warm-up by hand

1. Three trees predict the following votes for a test example: $\hat{y}^{(1)} = A$, $\hat{y}^{(2)} = A$, $\hat{y}^{(3)} = B$. Compute the forest prediction. Then repeat for votes $[B, B, A]$.
2. A bootstrap sample of size $n=8$ is drawn. The sampled indices are $[0, 1, 3, 3, 5, 6, 7, 7]$. Identify the OOB indices. Count how many distinct OOB samples there are out of $8$.

## Step 3: Implement the Random Forest core

Complete these TODOs in order. You may reuse your Lab 5 decision tree implementation; only the pieces below are new.

Implementation policy (strict):
- Implement the random forest core algorithm with NumPy only.
- Do not use a library random forest implementation for the main task.
- Plotting libraries are allowed for visual inspection.

### T1: Bootstrap sampler

Open `advanced_ai/forest/random_forest.py` and implement `_bootstrap_sample`.

Draw $n$ indices from $\{0,\dots,n-1\}$ with replacement and return the array of sampled indices.

### T2: Feature subsampler

Implement `_feature_subset`.

For a dataset with $d$ features, randomly select $m=\lfloor\sqrt{d}\rfloor$ feature indices and return them.

### T3: Bagging training loop

Implement `fit`.

For each of $B$ rounds, draw a bootstrap sample, create a new decision tree (reuse the class from Lab 5), train it on the bootstrapped data restricted to a random feature subset, and store the trained tree and the OOB index mask.

### T4: Prediction (classification)

Implement `predict`.

Each stored tree makes a prediction for the input; return the majority class across all trees.

### T5: Out-of-bag error

Implement `oob_error`.

For each training sample, find which trees had it OOB, aggregate the votes of those trees, compare the aggregated prediction with the true label, and return the fraction of OOB predictions that are wrong.

### T6: Feature importance

Implement `feature_importances`.

Walk through each tree and each internal node, sum $\frac{n_m}{n}\Delta I$ for each feature used as the split feature, average over $B$ trees, and normalize so the importances sum to 1.

## Step 4: Run the self-checks

After implementing the core class, run:

```bash
python -m advanced_ai.self_checks
```

Continue only after every check passes.

## Step 5: Toy Dataset

Run:

```bash
python toy_forest.py
```

Deliverables:
- depth, number of leaves, and training accuracy of a single tree;
- forest accuracy (same dataset) with $B=50$;
- saved decision-region comparison plot (single tree side by side with forest);
- one sentence explaining why the forest boundary is smoother.

## Step 6: Iris Data

Run:

```bash
python iris_forest.py
```

Deliverables:
- training accuracy, OOB error, and test accuracy for each $B \in \{1, 5, 10, 20, 50, 100, 200\}$;
- saved plot of OOB error and test accuracy versus $B$ on the same axes;
- the value of $B$ at which test accuracy plateaus.

## Step 7: OOB Error and Feature Importance

Run:

```bash
python oob_and_importance.py
```

Deliverables:
- final OOB error compared with test accuracy;
- top-3 features and their importance scores;
- whether the top features match what you know about the dataset.

## Step 8: Optional extension

Complete at most one optional extension after finishing the required work.

- Try different feature subset sizes: $m=1$, $m=\lfloor\sqrt{d}/2\rfloor$, $m=d$, and report how $m$ changes the OOB error curve.
- Try regression with a random forest on a simple 1D dataset.

## Submission checklist

Submit:
- completed `random_forests.ipynb`;
- completed `advanced_ai/forest/random_forest.py`;
- generated files from `results/`;
- one PDF report with hand calculations, plots, tables, and conclusions;
- reproducibility notes including random seed and environment.